## Setup

This notebook is inspired by: https://www.kaggle.com/code/andrewdblevins/leash-tutorial-ecfps-and-random-forest

In [2]:
pip install rdkit duckdb xgboost scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: /opt/homebrew/Cellar/jupyterlab/4.3.5/libexec/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Data Preparation

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

train_path = 'data/train.parquet'
test_path = 'data/test.parquet'

con = duckdb.connect()
df = con.query(f"""
    (SELECT * FROM parquet_scan('{train_path}') WHERE binds = 0 ORDER BY random() LIMIT 30000)
    UNION ALL
    (SELECT * FROM parquet_scan('{train_path}') WHERE binds = 1 ORDER BY random() LIMIT 30000)
""").df()
con.close()

df.head()

,id,buildingblock1_smiles,buildingblock2_smiles,buildingblock3_smiles,molecule_smiles,protein_name,binds
0,274780886,O=C(O)C[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)c1cc...,COc1c(N)cccc1F,Cc1nc(N)ccc1[N+](=O)[O-],COc1c(F)cccc1Nc1nc(Nc2ccc([N+](=O)[O-])c(C)n2)...,sEH,0
1,50869087,Cc1cc(C(=O)O)ccc1NC(=O)OCC1c2ccccc2-c2ccccc21,NCC1CCC(F)(F)CC1,Cl.NCC1CCC2(CCC2)CO1,Cc1cc(C(=O)N[Dy])ccc1Nc1nc(NCC2CCC(F)(F)CC2)nc...,HSA,0
2,174269168,O=C(Nc1cc(F)c(F)cc1C(=O)O)OCC1c2ccccc2-c2ccccc21,Nc1nc(Cl)ccc1Cl,Cl.Cl.NCC(=O)Nc1nccs1,O=C(CNc1nc(Nc2cc(F)c(F)cc2C(=O)N[Dy])nc(Nc2nc(...,sEH,0
3,246541046,O=C(O)C[C@@H](Cc1ccc(Cl)cc1)NC(=O)OCC1c2ccccc2...,CSc1ccc(CN)cc1C#N.Cl,Cc1nnc(N)o1,CSc1ccc(CNc2nc(Nc3nnc(C)o3)nc(N[C@@H](CC(=O)N[...,sEH,0
4,80506980,O=C(NC1(C(=O)O)CCOCC1)OCC1c2ccccc2-c2ccccc21,Nc1cccc(-n2cncn2)c1,CS(=O)(=O)c1ccc(N)cc1,CS(=O)(=O)c1ccc(Nc2nc(Nc3cccc(-n4cncn4)c3)nc(N...,BRD4,0


In [ ]:
df_sorted_test = df.sort_values(by='binds', ascending=False)

In [ ]:
df_sorted_test.head()

,id,buildingblock1_smiles,buildingblock2_smiles,buildingblock3_smiles,molecule_smiles,protein_name,binds
30000,47720449,Cc1cc(Br)c(C(=O)O)cc1NC(=O)OCC1c2ccccc2-c2ccccc21,Nc1cc(N2CCCC2)ccn1,CCOC(=O)c1c[nH]nc1N,CCOC(=O)c1c[nH]nc1Nc1nc(Nc2cc(N3CCCC3)ccn2)nc(...,HSA,1
40005,34678323,COc1ccc(C(=O)O)c(NC(=O)OCC2c3ccccc3-c3ccccc32)c1,Nc1ccc(F)c(Cl)c1,NCC1CSCCN1Cc1ccccc1,COc1ccc(C(=O)N[Dy])c(Nc2nc(NCC3CSCCN3Cc3ccccc3...,BRD4,1
39993,249543014,O=C(O)C[C@@H](Cc1ccc(I)cc1)NC(=O)OCC1c2ccccc2-...,CC(CN)S(=O)(=O)N1CCN(c2ccccc2)CC1.Cl.Cl,Cc1sc(N)c(C#N)c1C,Cc1sc(Nc2nc(NCC(C)S(=O)(=O)N3CCN(c4ccccc4)CC3)...,sEH,1
39994,64483026,N#Cc1ccc(C[C@@H](NC(=O)OCC2c3ccccc3-c3ccccc32)...,COC(C)(CCN)OC,COC(=O)Cc1nc(N)sc1C,COC(=O)Cc1nc(Nc2nc(NCCC(C)(OC)OC)nc(N[C@H](Cc3...,BRD4,1
39995,59917697,Cc1cccc(C(=O)O)c1NC(=O)OCC1c2ccccc2-c2ccccc21,Nc1nncs1,Nc1ncnc(=O)[nH]1,Cc1cccc(C(=O)N[Dy])c1Nc1nc(Nc2ncnc(=O)[nH]2)nc...,sEH,1


## Feature Preprocessing

In [ ]:
from rdkit import RDLogger
from sklearn.preprocessing import LabelEncoder

RDLogger.DisableLog('rdApp.*')

def generate_ecfp(smiles, radius=2, bits=1024):
    molecule = Chem.MolFromSmiles(smiles)
    return list(AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=bits)) if molecule else None

df['ecfp'] = df['molecule_smiles'].apply(generate_ecfp)
df.dropna(subset=['ecfp'], inplace=True)

onehot_encoder = OneHotEncoder(sparse_output=False)
protein_onehot = onehot_encoder.fit_transform(df['protein_name'].values.reshape(-1, 1))

X = np.array([ecfp + list(protein) for ecfp, protein in zip(df['ecfp'].tolist(), protein_onehot.tolist())])
y = df['binds'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Train Model

In [ ]:
import xgboost as xgb
from sklearn.metrics import average_precision_score

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

map_score = average_precision_score(y_test, y_pred_proba)
print(f"📈 Mean Average Precision (mAP): {map_score:.4f}")


📈 Mean Average Precision (mAP): 0.9652


## Predict Binding for New Molecules

In [ ]:
def predict_binding(smiles: str, protein_name: str):

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        return "Invalid SMILES"

    ecfp = generate_ecfp(smiles)

    protein_onehot = onehot_encoder.transform([[protein_name]])[0]

    X_new = np.array(ecfp + list(protein_onehot)).reshape(1, -1)

    probability = xgb_model.predict_proba(X_new)[0, 1]

    return probability

# Example Prediction
smiles_input = "O=C(C[C@H](Nc1nc(NCc2nccs2)nc(Nc2ccc3c(c2)CNC3=O)n1)c1cccc(Cl)c1Cl)N[Dy]"
protein_input = "BRD4"

predicted_prob = predict_binding(smiles_input, protein_input)
print(f"🔬 Predicted Binding Probability: {predicted_prob:.4f}")


🔬 Predicted Binding Probability: 0.9469


## Saving the pretrained model

In [ ]:
import joblib

joblib.dump(xgb_model, "xgb_model.pkl")
joblib.dump(onehot_encoder, "onehot_encoder.pkl")

xgb_model = joblib.load("xgb_model.pkl")
onehot_encoder = joblib.load("onehot_encoder.pkl")


## Loading and using the pretrained model

In [ ]:
import joblib
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

xgb_model = joblib.load("xgb_model.pkl")
onehot_encoder = joblib.load("onehot_encoder.pkl")

def generate_ecfp(molecule, radius=2, bits=1024):
    if molecule is None:
        return None
    return list(AllChem.GetMorganFingerprintAsBitVect(molecule, radius, nBits=bits))

def predict_single(smiles: str, protein_name: str):

    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError("❌ Invalid SMILES string")
    
    ecfp = generate_ecfp(molecule)
    if ecfp is None:
        raise ValueError("❌ Could not generate ECFP fingerprint")

    protein_onehot = onehot_encoder.transform([[protein_name]])[0]

    X_test = np.array(ecfp + list(protein_onehot)).reshape(1, -1)

    probability = xgb_model.predict_proba(X_test)[0, 1]

    return probability

# Example usage
if __name__ == "__main__":
    smiles_input = "O=C(C[C@H](Nc1nc(NCc2nccs2)nc(Nc2ccc3c(c2)CNC3=O)n1)c1cccc(Cl)c1Cl)N[Dy]"
    protein_input = "BRD4"
    
    predicted_prob = predict_single(smiles_input, protein_input)
    print(f"🔬 Predicted Binding Probability: {predicted_prob:.4f}")

🔬 Predicted Binding Probability: 0.9469


[16:26:51] DEPRECATION WARNING: please use MorganGenerator
